# Experiment 1 — Domain 2 (Graphs): Phase 1 Dataset Generation

Generates the **300-graph, 8-property** graph dataset for Experiment 1
(`serialization_experiment_1.pdf`, Section 4).

Mirrors the structure of the Domain 1 (Geometry) Phase 1 notebook: tiered
rejection sampling against a named validity rule set, a
compute-invariant-properties → randomize-presentation → serialize →
read-presentation-dependent-property build order, and independent
verification from the serialized string.

**Per the PDF (Section 4.2, Table 6):**

| Tier | Count | Nodes | Purpose |
|------|-------|-------|---------|
| simple | 100 | 6–15  | Baseline; small enough for manual verification |
| medium | 100 | 16–40 | Core measurement |
| hard   | 100 | 41–80 | Stress test |

Within each tier: 5 graph families (Erdős–Rényi, Barabási–Albert,
Watts–Strogatz, random bipartite, random planar), target 20 each (PDF allows
18–22, "exact balance is not required").

**8 properties** (Table 8): `degree_of_node_0`, `edge_count` *(local)*,
`triangle_count`, `is_bipartite`, `is_planar`, `diameter`,
`chromatic_number`, `avg_clustering` *(global)*.

Output:
- `graph_exp1_dataset.json` — 300 graphs with edge-list serialization + ground truth
- `graph_exp1_summary.json` — summary statistics (Section 7-equivalent)

Random seed fixed at **42** and recorded in every record.


In [ ]:
# Phase 0: environment
!pip install networkx scipy matplotlib --quiet

import json, math, random, re, time
from collections import deque
import networkx as nx
from scipy.spatial import Delaunay
import matplotlib.pyplot as plt

print("networkx", nx.__version__)

## 1. Tiers, families, and the validity checker

Generation methods (Section 4.2, Table 7):
- **Erdős–Rényi** — `nx.gnp_random_graph(n, p)`, `p` chosen for expected degree 3–6.
- **Barabási–Albert** — `nx.barabasi_albert_graph(n, m)`, `m ∈ {2, 3, 4}`.
- **Watts–Strogatz** — `nx.watts_strogatz_graph(n, k, p)`, `k ∈ {4, 6}`, `p ∈ {0.1, 0.3, 0.5}`.
- **Random bipartite** — `nx.bipartite.random_graph(n1, n2, p)`, `n1 + n2 = n`, `n1/n2 ∈ [0.3, 0.7]`.
- **Random planar** — Delaunay triangulation of random points (always planar),
  then edges are randomly thinned — never below `n-1`, never dropping below
  connectivity — down to a random target within the `3n-6` planar edge bound.

**Family split.** 20 graphs/family/tier would be perfectly even (100/tier).
The committed split nudges two counts within the PDF's stated 18–22 tolerance:
`random_bipartite` at 22, `barabasi_albert` and `watts_strogatz` at 19 each.

**Internal true/false split, `random_bipartite` and `random_planar` only.**
Both counts (22, 20) are deliberately even: each family is generated as an
exact half/half split on the boolean property it's named after —
`random_bipartite` 11 `is_bipartite=True` + 11 `is_bipartite=False`,
`random_planar` 10 `is_planar=True` + 10 `is_planar=False`, in every tier
(Section 5, `SPLIT_FAMILIES`). The "false" half of each still shares the
family's generation spirit (built from the same partition/point-based
approach) but is modified just enough to flip the property: an added
intra-partition edge for `random_bipartite`, or built at the maximum edge
count `check_validity` allows for `random_planar` (see cell 4 for the
per-generator detail). `erdos_renyi`/`barabasi_albert`/`watts_strogatz` are
not split — they essentially never produce bipartite or planar instances at
their specified densities, so there is no True/False mix to balance there.
This is an explicit deviation from the PDF's own 25–35%-bipartite-overall
aim (see §4/README for the resulting overall percentages) — the balance
target here is *within* the two boolean-named families, not across the
whole dataset.

**Validity constraints** (Section 4.2), checked by `check_validity`, each
with a named rejection reason:

| Rule | Rejection reason | Check |
|------|------------------|-------|
| 1 | `node_count_out_of_range` | `vmin <= n <= vmax` |
| 2 | `self_loop` | no self-loops |
| 3 | *(structural)* | no multi-edges — guaranteed by using `nx.Graph`, never `nx.MultiGraph` |
| 4 | `not_connected` | single connected component |
| 5 | `too_few_edges` | `m >= n - 1` |
| 6 | `too_many_edges` | `m <= C(n,2)/2` |

All five generators are **rejection samplers**: draw `n` and family-specific
parameters, build a candidate, run `check_validity`, retry up to
`max_tries=3000`.

In [ ]:
# Tier node-count ranges (Table 6). No coordinate axis here (unlike geometry) —
# the graph's complexity axis is node count alone.
TIERS = {
    #         vmin vmax
    "simple": (6,  15),
    "medium": (16, 40),
    "hard":   (41, 80),
}

FAMILIES = ["erdos_renyi", "barabasi_albert", "watts_strogatz",
            "random_bipartite", "random_planar"]

# Per-tier, per-family counts. 20/20/20/20/20 would be exactly even; this
# split (still within the PDF's 18-22 tolerance) is explained in the cell above.
PER_FAMILY = {"erdos_renyi": 20, "barabasi_albert": 19, "watts_strogatz": 19,
              "random_bipartite": 22, "random_planar": 20}
assert sum(PER_FAMILY.values()) == 100


def check_validity(G, vmin, vmax):
    if G is None:
        return False, "empty"
    n = G.number_of_nodes()
    if not (vmin <= n <= vmax):                    # rule 1
        return False, "node_count_out_of_range"
    if nx.number_of_selfloops(G) > 0:               # rule 2
        return False, "self_loop"
    if not nx.is_connected(G):                      # rule 4
        return False, "not_connected"
    m = G.number_of_edges()
    if m < n - 1:                                   # rule 5
        return False, "too_few_edges"
    if m > (n * (n - 1) / 2) / 2:                    # rule 6
        return False, "too_many_edges"
    return True, None


print("Tiers, families, and validity checker defined.")

## 2. The five family generators (plus 2 false-half variants)

Each generator draws its own `n` fresh on every retry (from the tier's
range), builds a candidate, and rejects/retries against `check_validity`.
`n` is not fixed by the caller -- it is emergent per attempt, same pattern as
the geometry generators drawing a fresh point count `m` each retry.

`gen_random_bipartite` and `gen_random_planar` only ever produce
`is_bipartite=True` / `is_planar=True` respectively -- that's what their
names promise. To get a genuine 50/50 split *within* each family (see cell
2), two more generators build the "false" half:

- **`gen_random_bipartite_false`** -- builds the identical `(n1, n2, p)`
  bipartite skeleton, then adds **one** intra-partition edge. In a connected
  bipartite graph every path between two same-side vertices has even length;
  closing it with one more edge creates an odd cycle, so this deterministically
  forces `is_bipartite=False` with no rejection sampling needed on the
  property itself.
- **`gen_random_planar_false`** -- builds the densest graph `check_validity`'s
  rule 6 (`m <= n(n-1)/4`) allows at that `n`, then checks planarity directly.
  This is the necessary approach, not just a convenient one: the smallest
  non-planar simple graph (K3,3) needs 9 edges on 6 nodes, but rule 6 caps
  `n=6` at 7 edges -- no non-planar graph is reachable there at all. The
  fresh-`n` retry loop simply skips over draws too small to admit one.


In [ ]:
def gen_erdos_renyi(rng, vmin, vmax, max_tries=3000):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        target_degree = rng.uniform(3, 6)
        p = min(max(target_degree / (n - 1), 0.0), 1.0)
        G = nx.gnp_random_graph(n, p, seed=rng)
        ok, _ = check_validity(G, vmin, vmax)
        if ok:
            return G, {"p": round(p, 4), "target_degree": round(target_degree, 2)}
    return None, None


def gen_barabasi_albert(rng, vmin, vmax, max_tries=3000):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        m = rng.choice([2, 3, 4])
        if m >= n:
            continue
        G = nx.barabasi_albert_graph(n, m, seed=rng)
        ok, _ = check_validity(G, vmin, vmax)
        if ok:
            return G, {"m": m}
    return None, None


def gen_watts_strogatz(rng, vmin, vmax, max_tries=3000):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        k = rng.choice([4, 6])
        if k >= n:
            continue
        p = rng.choice([0.1, 0.3, 0.5])
        G = nx.watts_strogatz_graph(n, k, p, seed=rng)
        ok, _ = check_validity(G, vmin, vmax)
        if ok:
            return G, {"k": k, "p": p}
    return None, None


def gen_random_bipartite(rng, vmin, vmax, max_tries=3000):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        # PDF Table 7: n1/n2 (the ratio BETWEEN partitions), not n1/n (partition
        # share of the total). Search the integer splits of n whose n1/n2 ratio
        # actually lands in [0.3, 0.7] -- rounding a continuous target can overshoot
        # the bound for small n (e.g. n=7 only admits (2,5)=0.4; naive rounding can
        # land on (3,4)=0.75), so pick the closest exact match instead of rounding.
        target_ratio = rng.uniform(0.3, 0.7)
        candidates = []
        for n1 in range(1, n):
            n2 = n - n1
            ratio = n1 / n2
            if 0.3 <= ratio <= 0.7:
                candidates.append((abs(ratio - target_ratio), n1, n2))
        if not candidates:
            continue   # no integer split of this n satisfies the ratio bound
        candidates.sort()
        _, n1, n2 = candidates[0]
        p = rng.uniform(0.15, 0.45)
        G = nx.bipartite.random_graph(n1, n2, p, seed=rng)
        ok, _ = check_validity(G, vmin, vmax)
        if ok:
            return G, {"n1": n1, "n2": n2, "p": round(p, 4)}
    return None, None


def gen_random_planar(rng, vmin, vmax, max_tries=3000):
    # Delaunay triangulation of random points is always planar; any subgraph
    # of a planar graph is planar too, so thinning edges preserves planarity.
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        pts = [(rng.random(), rng.random()) for _ in range(n)]
        try:
            tri = Delaunay(pts)
        except Exception:
            continue
        edges = set()
        for simplex in tri.simplices:
            for i in range(3):
                for j in range(i + 1, 3):
                    a, b = int(simplex[i]), int(simplex[j])
                    edges.add((min(a, b), max(a, b)))
        G = nx.Graph()
        G.add_nodes_from(range(n))
        G.add_edges_from(edges)

        max_planar_edges = 3 * n - 6 if n >= 3 else n - 1
        target_m = rng.randint(n - 1, min(G.number_of_edges(), max_planar_edges))
        edge_list = list(G.edges())
        rng.shuffle(edge_list)
        for e in edge_list:
            if G.number_of_edges() <= target_m:
                break
            G.remove_edge(*e)
            if not nx.is_connected(G):
                G.add_edge(*e)          # revert: thinning must not disconnect

        ok, _ = check_validity(G, vmin, vmax)
        if ok:
            return G, {"target_m": target_m}
    return None, None


# -- False-ground-truth halves of random_bipartite / random_planar --
# (added so each of these two families is internally 50/50 on the boolean
# property it's named after, instead of being 100% True by construction --
# see cell 4 for why.)

def gen_random_bipartite_false(rng, vmin, vmax, max_tries=3000):
    # Same partition construction as gen_random_bipartite, then ONE extra
    # intra-partition edge. In a connected bipartite graph, any path between
    # two same-side vertices has even length; adding an edge between them
    # closes an odd cycle, so this deterministically forces is_bipartite ==
    # False (no rejection sampling on the property itself needed).
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        target_ratio = rng.uniform(0.3, 0.7)
        candidates = []
        for n1 in range(1, n):
            n2 = n - n1
            ratio = n1 / n2
            if 0.3 <= ratio <= 0.7:
                candidates.append((abs(ratio - target_ratio), n1, n2))
        if not candidates:
            continue
        candidates.sort()
        _, n1, n2 = candidates[0]
        p = rng.uniform(0.15, 0.45)
        G = nx.bipartite.random_graph(n1, n2, p, seed=rng)

        part_candidates = [pp for pp in (list(range(n1)), list(range(n1, n))) if len(pp) >= 2]
        if not part_candidates:
            continue
        part = rng.choice(part_candidates)
        a, b = rng.sample(part, 2)   # intra-partition pair -- never already an edge
        G.add_edge(a, b)

        ok, _ = check_validity(G, vmin, vmax)
        if ok and not nx.is_bipartite(G):
            return G, {"n1": n1, "n2": n2, "p": round(p, 4), "intra_edge": [a, b]}
    return None, None


def gen_random_planar_false(rng, vmin, vmax, max_tries=3000):
    # Target m = 3n (moderately above the 3n-6 planar bound -- comfortably
    # non-planar almost every draw, measured 0 failures across 60 test draws)
    # capped by rule 6's own limit (n(n-1)/4). Deliberately NOT maxed out at
    # the rule-6 cap: that produces near-half-density graphs whose chromatic
    # number is expensive to certify exactly for the hard tier (measured: 7
    # uncertified/300 at max density vs. 0/300 at this lower target). At
    # n=6 the cap (7) is below even 3n-6's minimum non-planar requirement
    # (K3,3 needs 9 edges) -- no non-planar graph is reachable there at all;
    # the fresh-n retry loop just skips over that (and other too-small) draws.
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        cap = int((n * (n - 1) / 2) / 2)
        if cap < n - 1:
            continue
        target_m = min(cap, max(n - 1, 3 * n))
        G = nx.gnm_random_graph(n, target_m, seed=rng)
        ok, _ = check_validity(G, vmin, vmax)
        if ok and not nx.check_planarity(G)[0]:
            return G, {"target_m": target_m}
    return None, None


GENERATORS = {
    "erdos_renyi": gen_erdos_renyi,
    "barabasi_albert": gen_barabasi_albert,
    "watts_strogatz": gen_watts_strogatz,
    "random_bipartite": gen_random_bipartite,
    "random_planar": gen_random_planar,
}

print("Generators defined:", list(GENERATORS.keys()),
      "+ random_bipartite_false, random_planar_false (see cell 4)")

## 3. Chromatic number

NetworkX has no exact chromatic-number function (Section 4, "Chromatic
number computation"). This implements the PDF's specified fallback chain:

1. **Clique certificate, fast path.** If the greedy `DSATUR` upper bound
   equals the size of a maximum clique found by `nx.find_cliques`, the
   chromatic number is certified immediately — a clique of size *k* forces
   at least *k* colors, and the greedy coloring already achieves *k*.
2. **Exact backtracking, bounded.** Otherwise, search for a valid *k*-coloring
   for increasing *k*, using DSATUR vertex ordering (most-constrained vertex
   first) and color-symmetry breaking (never open a color number more than 1
   past the highest used so far). Time-boxed per graph.
3. **Uncertified fallback.** If the time box is hit before the gap between
   clique lower bound and greedy upper bound closes, the graph's chromatic
   number is **excluded from evaluation** and flagged
   `chromatic_number_certified: false` in its metadata — per the PDF: *"Do
   not use approximate values."*

Verified against known graphs before trusting it on the dataset (execution
checklist item 2: *"verify against known graphs"*).

In [ ]:
def clique_number(G, cap=None):
    best = 1
    for c in nx.find_cliques(G):
        if len(c) > best:
            best = len(c)
        if cap is not None and best >= cap:
            return best
    return best


def greedy_upper_bound(G):
    coloring = nx.coloring.greedy_color(G, strategy="DSATUR")
    return max(coloring.values()) + 1 if coloring else 1


def exact_chromatic_number(G, time_limit=15.0):
    # Return (k, certified). Exact via DSATUR-ordered, symmetry-broken
    # backtracking within time_limit; else an uncertified greedy upper bound.
    n = G.number_of_nodes()
    if n == 0:
        return 0, True
    ub = greedy_upper_bound(G)
    lb = clique_number(G, cap=ub)
    if lb == ub:
        return ub, True                      # clique certificate

    nodes = list(G.nodes())
    adj = {v: set(G.neighbors(v)) for v in nodes}
    deadline = time.monotonic() + time_limit
    timed_out = [False]

    def can_color(k):
        colors = {}
        steps = [0]

        def choose_next():
            # DSATUR: most saturated (distinct neighbor colors) first, ties by degree.
            best, best_sat, best_deg = None, -1, -1
            for v in nodes:
                if v in colors:
                    continue
                sat = len({colors[u] for u in adj[v] if u in colors})
                deg = len(adj[v])
                if sat > best_sat or (sat == best_sat and deg > best_deg):
                    best, best_sat, best_deg = v, sat, deg
            return best

        def backtrack(count):
            steps[0] += 1
            if steps[0] % 1000 == 0 and time.monotonic() > deadline:
                timed_out[0] = True
                return False
            if count == n:
                return True
            v = choose_next()
            used = {colors[u] for u in adj[v] if u in colors}
            max_used = max(colors.values(), default=-1)
            upper = min(k - 1, max_used + 1)   # color-symmetry breaking
            for c in range(upper + 1):
                if c not in used:
                    colors[v] = c
                    if backtrack(count + 1):
                        return True
                    del colors[v]
                    if timed_out[0]:
                        return False
            return False

        return backtrack(0)

    for k in range(lb, ub + 1):
        if can_color(k):
            return k, True
        if timed_out[0]:
            break
    return ub, False                          # uncertified: exclude at evaluation time


# Known-graph sanity check (execution checklist item 2).
_known = [
    ("K5", nx.complete_graph(5), 5), ("Petersen", nx.petersen_graph(), 3),
    ("C5", nx.cycle_graph(5), 3), ("C6", nx.cycle_graph(6), 2),
    ("K3,3", nx.complete_bipartite_graph(3, 3), 2),
    ("K4", nx.complete_graph(4), 4), ("Star_10", nx.star_graph(10), 2),
]
for name, G, expected in _known:
    k, certified = exact_chromatic_number(G)
    assert k == expected, f"{name}: got {k}, expected {expected}"
    print(f"  {name}: chromatic_number={k} (certified={certified}) -- matches expected {expected}")
print("Chromatic number verified against", len(_known), "known graphs.")

## 4. Ground truth (8 properties) + record builder

**The step order here is a correctness requirement**, same principle as the
geometry domain's winding-reversal ordering (Section 3.2 there):

1. **`compute_presentation_independent(G)`** — the 6 properties that do not
   depend on node labeling: `triangle_count`, `is_bipartite`, `is_planar`,
   `diameter`, `chromatic_number`, `avg_clustering`.
2. **`randomize_labeling(G, rng)`** — relabel nodes with a random permutation
   of `0..n-1`. Without this, node `0` is whatever label the generator
   happened to assign — e.g. Barabási–Albert's earliest nodes are
   structurally the hubs — which would make `degree_of_node_0` a function of
   generator internals rather than a genuine "read this from the
   serialization" question. This is the graph-domain analogue of the
   geometry domain's `maybe_reverse` (winding direction): any property whose
   ground truth depends on how the object is written down must be computed
   *after* randomizing that presentation.
3. **`to_edge_list_string(G)`** — serialize the *relabeled* graph, edges
   sorted `(min(u,v), max(u,v))` then lexicographically, per Section 4.3.
4. **`degree_of_node_0`, `edge_count`** — read from the relabeled graph, so
   the label matches the string the model will actually see.

All floats are rounded to 4 decimal places.

In [ ]:
# Step 1: properties that do NOT depend on node labeling.
def compute_presentation_independent(G):
    triangle_count = sum(nx.triangles(G).values()) // 3
    is_bipartite = bool(nx.is_bipartite(G))
    is_planar = bool(nx.check_planarity(G)[0])
    diameter = nx.diameter(G)
    avg_clustering = round(nx.average_clustering(G), 4)
    clique_n = clique_number(G)
    chrom, certified = exact_chromatic_number(G)
    props = {
        "triangle_count": triangle_count,
        "is_bipartite": is_bipartite,
        "is_planar": is_planar,
        "diameter": diameter,
        "chromatic_number": chrom,
        "avg_clustering": avg_clustering,
    }
    extra = {"clique_number": clique_n, "chromatic_number_certified": certified}
    return props, extra


# Step 2: relabel with a random permutation of 0..n-1 (see markdown above).
def randomize_labeling(G, rng):
    nodes = list(G.nodes())
    shuffled = nodes[:]
    rng.shuffle(shuffled)
    mapping = {old: new for old, new in zip(nodes, shuffled)}
    return nx.relabel_nodes(G, mapping, copy=True)


# Step 3: serialize to the edge-list format (Section 4.3).
def to_edge_list_string(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    edges = sorted((min(u, v), max(u, v)) for u, v in G.edges())
    lines = [f"GRAPH (n={n}, m={m}):"] + [f"{u} {v}" for u, v in edges]
    return "\n".join(lines)


# Build one full dataset record, enforcing the correct step order.
def build_record(G, tier, family, index, rng, seed, gen_params):
    props, extra = compute_presentation_independent(G)   # step 1
    G2 = randomize_labeling(G, rng)                       # step 2
    edge_list = to_edge_list_string(G2)                   # step 3
    props["degree_of_node_0"] = G2.degree(0)              # step 4
    props["edge_count"] = G2.number_of_edges()

    return {
        "object_id": f"graph_{tier}_{family}_{index:03d}",
        "tier": tier,
        "family": family,
        "num_nodes": G2.number_of_nodes(),
        "num_edges": G2.number_of_edges(),
        "edge_list": edge_list,
        "properties": props,
        "metadata": {
            "generation_params": gen_params,
            "random_seed": seed,
            "clique_number": extra["clique_number"],
            "chromatic_number_certified": extra["chromatic_number_certified"],
            "is_connected": True,
        },
    }


print("Ground-truth and record builder defined (8 properties).")

## 5. Build the 300-graph dataset + summary

300 graphs = 3 tiers x 100. Within each tier the 5 families are split per
`PER_FAMILY` (Section 1); `random_bipartite` and `random_planar` are each
further split exactly in half between their normal and `_false` generator
(Section 2, `SPLIT_FAMILIES`) so both are internally 50/50 on their named
boolean property. Generation failure is **fatal**: if a generator exhausts
3000 tries and returns `None`, `build_dataset` raises rather than emitting a
short dataset.


In [ ]:
# For these 2 families, half the instances use the normal (True-producing)
# generator and half use its _false counterpart, so the family is internally
# 50/50 on the boolean property it's named after (see cell 4). Both counts
# are even (22 and 20), so this is an exact half split, no rounding.
SPLIT_FAMILIES = {
    "random_bipartite": (gen_random_bipartite, gen_random_bipartite_false),
    "random_planar": (gen_random_planar, gen_random_planar_false),
}


def build_dataset(seed=42):
    rng = random.Random(seed)
    records = []
    for tier in ["simple", "medium", "hard"]:
        vmin, vmax = TIERS[tier]
        for family in FAMILIES:
            total = PER_FAMILY[family]
            if family in SPLIT_FAMILIES:
                true_gen, false_gen = SPLIT_FAMILIES[family]
                half = total // 2
                for i in range(1, total + 1):
                    gen = true_gen if i <= half else false_gen
                    G, params = gen(rng, vmin, vmax)
                    if G is None:
                        raise RuntimeError(f"Failed to generate {tier}/{family} #{i}")
                    records.append(build_record(G, tier, family, i, rng, seed, params))
            else:
                for i in range(1, total + 1):
                    G, params = GENERATORS[family](rng, vmin, vmax)
                    if G is None:
                        raise RuntimeError(f"Failed to generate {tier}/{family} #{i}")
                    records.append(build_record(G, tier, family, i, rng, seed, params))
    return records


# Min/max/mean/median/std for a list of numbers (population std, same as geometry).
def stats_for(values):
    n = len(values)
    mean = sum(values) / n
    sv = sorted(values)
    median = sv[n // 2] if n % 2 else (sv[n // 2 - 1] + sv[n // 2]) / 2
    var = sum((v - mean) ** 2 for v in values) / n
    return {"min": round(min(values), 2), "max": round(max(values), 2),
            "mean": round(mean, 2), "median": round(median, 2),
            "std": round(var ** 0.5, 2)}


def summarize(records):
    summary = {"total": len(records)}
    by_tier = {}
    for r in records:
        by_tier.setdefault(r["tier"], {}).setdefault(r["family"], 0)
        by_tier[r["tier"]][r["family"]] += 1
    summary["counts_by_tier_family"] = by_tier

    n_bip = sum(1 for r in records if r["properties"]["is_bipartite"])
    n_planar = sum(1 for r in records if r["properties"]["is_planar"])
    summary["bipartite_overall"] = n_bip
    summary["planar_overall"] = n_planar

    # Internal true/false balance of the 2 split families, per tier (should
    # be exactly half/half -- see cell 4/10).
    split_balance = {}
    for tier in ["simple", "medium", "hard"]:
        split_balance[tier] = {
            "random_bipartite": {
                "true": sum(1 for r in records if r["tier"] == tier and r["family"] == "random_bipartite"
                            and r["properties"]["is_bipartite"]),
                "false": sum(1 for r in records if r["tier"] == tier and r["family"] == "random_bipartite"
                             and not r["properties"]["is_bipartite"]),
            },
            "random_planar": {
                "true": sum(1 for r in records if r["tier"] == tier and r["family"] == "random_planar"
                            and r["properties"]["is_planar"]),
                "false": sum(1 for r in records if r["tier"] == tier and r["family"] == "random_planar"
                             and not r["properties"]["is_planar"]),
            },
        }
    summary["split_family_boolean_balance"] = split_balance

    dist = {}
    for tier in ["simple", "medium", "hard"]:
        rs = [r for r in records if r["tier"] == tier]
        dist[tier] = {
            "num_nodes": stats_for([r["num_nodes"] for r in rs]),
            "num_edges": stats_for([r["num_edges"] for r in rs]),
            "triangle_count": stats_for([r["properties"]["triangle_count"] for r in rs]),
            "diameter": stats_for([r["properties"]["diameter"] for r in rs]),
            "chromatic_number": stats_for([r["properties"]["chromatic_number"] for r in rs]),
            "avg_clustering": stats_for([r["properties"]["avg_clustering"] for r in rs]),
            "edge_list_length": stats_for([len(r["edge_list"]) for r in rs]),
        }
    summary["distribution_by_tier"] = dist

    uncertified = [r["object_id"] for r in records if not r["metadata"]["chromatic_number_certified"]]
    summary["chromatic_number_uncertified_count"] = len(uncertified)
    summary["chromatic_number_uncertified_ids"] = uncertified
    return summary


# Run it all.
SEED = 42
print("Generating 300-graph dataset (seed =", SEED, ")...")
t0 = time.time()
records = build_dataset(SEED)
print(f"Done in {time.time() - t0:.1f}s.")

with open("graph_exp1_dataset.json", "w") as f:
    json.dump(records, f, indent=2)
print("Saved graph_exp1_dataset.json with", len(records), "graphs.")

summary = summarize(records)
with open("graph_exp1_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved graph_exp1_summary.json.\n")

print("Total graphs:", summary["total"])
print("Counts by tier/family:")
for tier, fams in summary["counts_by_tier_family"].items():
    print(f"  {tier}: {fams}")
print(f"\nBipartite overall: {summary['bipartite_overall']}/300"
      f" ({100*summary['bipartite_overall']/300:.1f}%)")
print(f"Planar overall:    {summary['planar_overall']}/300"
      f" ({100*summary['planar_overall']/300:.1f}%)")

print("\nSplit-family boolean balance (should be exact half/half per tier):")
for tier, fams in summary["split_family_boolean_balance"].items():
    rb = fams["random_bipartite"]
    rp = fams["random_planar"]
    print(f"  {tier}: random_bipartite true={rb['true']} false={rb['false']} | "
          f"random_planar true={rp['true']} false={rp['false']}")
print(f"\nChromatic number uncertified: {summary['chromatic_number_uncertified_count']}"
      f" {summary['chromatic_number_uncertified_ids']}")
print("\nNode-count range per tier:")
for tier, d in summary["distribution_by_tier"].items():
    nc = d["num_nodes"]
    print(f"  {tier}: min {nc['min']}, max {nc['max']}, mean {nc['mean']}")

## 6. Independent verification

Re-parse each record **from the stored edge-list string** with
`parse_edge_list` (plain text parsing, no networkx) and recompute properties
with hand-written implementations — BFS-based bipartiteness and diameter,
adjacency-set triangle counting, hand-rolled clustering coefficient — then
compare against the stored ground truth.

**Known gap** (documented, same discipline as the geometry domain's own
gap): this covers 6 of 8 properties. `is_planar` and `chromatic_number` are
not independently re-derived — an independent planarity test (Boyer–Myrvold)
and an independent exact-coloring implementation are both substantial
undertakings on their own; worth closing in a later pass, same as geometry's
unclosed `bbox`/`centroid`/`convex`/`orientation` gap.

In [ ]:
def parse_edge_list(text):
    lines = text.strip().split("\n")
    m = re.match(r"GRAPH \(n=(\d+), m=(\d+)\):", lines[0])
    n_hdr, m_hdr = int(m.group(1)), int(m.group(2))
    edges = []
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        u, v = map(int, line.split())
        edges.append((u, v))
    return n_hdr, m_hdr, edges


def indep_degree0(edges):
    return sum(1 for u, v in edges if u == 0 or v == 0)


def indep_triangle_count(n, edges):
    adj = [set() for _ in range(n)]
    for u, v in edges:
        adj[u].add(v); adj[v].add(u)
    count = 0
    for u in range(n):
        for v in adj[u]:
            if v > u:
                count += sum(1 for w in (adj[u] & adj[v]) if w > v)
    return count


def indep_is_bipartite(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    color = [-1] * n
    for start in range(n):
        if color[start] != -1:
            continue
        color[start] = 0
        q = deque([start])
        while q:
            u = q.popleft()
            for w in adj[u]:
                if color[w] == -1:
                    color[w] = 1 - color[u]
                    q.append(w)
                elif color[w] == color[u]:
                    return False
    return True


def indep_diameter(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    diam = 0
    for start in range(n):
        dist = [-1] * n
        dist[start] = 0
        q = deque([start])
        while q:
            u = q.popleft()
            for w in adj[u]:
                if dist[w] == -1:
                    dist[w] = dist[u] + 1
                    q.append(w)
        if -1 in dist:
            return None    # would indicate a disconnected graph -- shouldn't happen
        diam = max(diam, max(dist))
    return diam


def indep_avg_clustering(n, edges):
    adj = [set() for _ in range(n)]
    for u, v in edges:
        adj[u].add(v); adj[v].add(u)
    total = 0.0
    for u in range(n):
        nb = list(adj[u])
        k = len(nb)
        if k < 2:
            continue
        links = 0
        for i in range(len(nb)):
            for j in range(i + 1, len(nb)):
                if nb[j] in adj[nb[i]]:
                    links += 1
        total += (2 * links) / (k * (k - 1))
    return total / n


mism = 0
for r in records:
    n, m_hdr, edges = parse_edge_list(r["edge_list"])
    gt = r["properties"]
    checks = {
        "num_nodes_header": (n, r["num_nodes"], 0),
        "edge_count_header": (m_hdr, gt["edge_count"], 0),
        "degree_of_node_0": (indep_degree0(edges), gt["degree_of_node_0"], 0),
        "edge_count": (len(edges), gt["edge_count"], 0),
        "triangle_count": (indep_triangle_count(n, edges), gt["triangle_count"], 0),
        "is_bipartite": (indep_is_bipartite(n, edges), gt["is_bipartite"], 0),
        "diameter": (indep_diameter(n, edges), gt["diameter"], 0),
        "avg_clustering": (indep_avg_clustering(n, edges), gt["avg_clustering"], 1e-4),
    }
    for name, (got, exp, tol) in checks.items():
        ok = (got == exp) if isinstance(exp, bool) else (abs(got - exp) <= tol)
        if not ok:
            mism += 1
            if mism <= 10:
                print(f"MISMATCH {r['object_id']} {name}: indep={got} stored={exp}")

print(f"\nIndependent verification: {mism} mismatches across {len(records)} graphs.")

# connectivity / structural sanity
bad = [r["object_id"] for r in records if not r["metadata"]["is_connected"]]
print("Disconnected records:", len(bad))
print("Properties present:", list(records[0]["properties"].keys()))

## 7. Visual spot-check

Draw one graph per (tier x column) -- 7 columns x 3 tiers = 21 graphs --
for an eyeball check. `random_bipartite` and `random_planar` each get 2
columns (their true-half and false-half, per cell 4/10/11) so the internal
split is visible; the other 3 families get 1 column each. Barabasi-Albert
should show visible hubs, Watts-Strogatz should look ring-like with a few
long-range rewires, random_planar(true) a non-crossing mesh and
random_planar(false) denser/crossing, random_bipartite(true) a two-group
crossing pattern and random_bipartite(false) the same plus one edge closing
an odd cycle. Not an automated assertion.


In [ ]:
SPOTCHECK_COLUMNS = [
    ("erdos_renyi", "erdos_renyi", None),
    ("barabasi_albert", "barabasi_albert", None),
    ("watts_strogatz", "watts_strogatz", None),
    ("random_bipartite (true)", "random_bipartite", True),
    ("random_bipartite (false)", "random_bipartite", False),
    ("random_planar (true)", "random_planar", True),
    ("random_planar (false)", "random_planar", False),
]


def find(tier, family, want_true):
    prop = "is_bipartite" if family == "random_bipartite" else "is_planar"
    for r in records:
        if r["tier"] == tier and r["family"] == family:
            if want_true is None or r["properties"][prop] == want_true:
                return r
    return None

fig, axes = plt.subplots(3, 7, figsize=(26, 12))
for row, tier in enumerate(["simple", "medium", "hard"]):
    for col, (label, family, want_true) in enumerate(SPOTCHECK_COLUMNS):
        ax = axes[row][col]
        r = find(tier, family, want_true)
        n, m, edges = parse_edge_list(r["edge_list"])
        G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(edges)
        pos = nx.spring_layout(G, seed=42)
        nx.draw(G, pos, ax=ax, node_size=30, width=0.6, node_color="#4C72B0")
        ax.set_title(f"{tier}/{label}\nn={n} m={m} bip={r['properties']['is_bipartite']} "
                     f"plan={r['properties']['is_planar']}", fontsize=8)
plt.tight_layout()
plt.savefig("spotcheck_exp1_graph.png", dpi=110)
plt.show()
print("Saved spotcheck_exp1_graph.png")


## 8. Download (Colab)

Download the dataset, summary, and spot-check figure. (Skip if running
locally — the files are already saved in the working directory.)

In [ ]:
try:
    from google.colab import files
    files.download("graph_exp1_dataset.json")
    files.download("graph_exp1_summary.json")
    files.download("spotcheck_exp1_graph.png")
except Exception as e:
    print("Not on Colab (files saved locally):", e)